# Статистика по экспериментам

Загружает `grid_search.csv` из указанной папки и считает по метрикам:
- среднее
- 25-й перцентиль
- 75-й перцентиль

**Колонки результата:**
- `score_samples` — на каком **n-м наблюдении** на диск скорили test (из `score_sample_grid`). Это не bootstrap.
- `n_bootstrap` — сколько **bootstrap-итераций** было для оценки разброса метрик (из `bootstrap_n`). Появляется только если bootstrap включён.

Поддерживаются:
- одна папка эксперимента (`Artifacts/Exp_164`)
- родительская папка с несколькими `Exp_*`
- bootstrap-результаты (несколько строк на `model_id` с `bootstrap_iter`)

Укажите путь в следующей ячейке и запустите все ячейки.

In [1]:
from pathlib import Path

import pandas as pd

# Путь до папки эксперимента или родительской папки (например Artifacts или Artifacts/Exp_164)
EXPERIMENTS_PATH = "Artifacts/Exp_166"

# Какие метрики показывать: только test, или все (*_test, *_train_same_size, *_train_max_size)
METRIC_SUFFIX = "all"  # "_test" | "all"

# Показывать только успешные запуски
ONLY_SUCCESS = True

In [2]:
from pathlib import Path
from typing import List, Union

import pandas as pd

RESULT_FILENAMES = ("grid_search.csv", "eval_grid_search.csv")


def resolve_repo_path(path: Union[str, Path]) -> Path:
    p = Path(path).expanduser()
    if not p.is_absolute():
        repo_root = Path.cwd()
        for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
            if (candidate / "disk_analyzer").exists():
                repo_root = candidate
                break
        p = repo_root / p
    return p.resolve()


def discover_result_files(root: Path) -> List[Path]:
    root = root.resolve()
    if not root.exists():
        raise FileNotFoundError(f"Папка не найдена: {root}")

    for name in RESULT_FILENAMES:
        direct = root / name
        if direct.exists():
            return [direct]

    files: List[Path] = []
    for exp_dir in sorted(root.glob("Exp_*")):
        if not exp_dir.is_dir():
            continue
        for name in RESULT_FILENAMES:
            candidate = exp_dir / name
            if candidate.exists():
                files.append(candidate)
                break

    if not files:
        raise FileNotFoundError(
            f"Не найдено grid_search.csv в {root} и подпапках Exp_*"
        )
    return files


def load_experiments(root: Path) -> pd.DataFrame:
    frames = []
    for csv_path in discover_result_files(root):
        df = pd.read_csv(csv_path)
        df["exp_name"] = csv_path.parent.name
        df["result_file"] = csv_path.name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def metric_columns(df: pd.DataFrame, suffix: str) -> List[str]:
    known_suffixes = ("_test", "_train_same_size", "_train_max_size")
    cols = [c for c in df.columns if any(c.endswith(s) for s in known_suffixes)]
    if suffix == "_test":
        cols = [c for c in cols if c.endswith("_test")]
    return sorted(cols)


PRIMARY_GROUP_COLS = ("exp_name", "method", "model_id", "test_samples", "score_samples", "train_samples")
SUMMARY_META_COLS = ("n_bootstrap",)
EXCLUDED_COLS = {
    "train_time", "test_time", "error", "error_text",
    "bootstrap_iter", "result_file",
}


def hyperparam_columns(df: pd.DataFrame) -> List[str]:
    primary = [c for c in PRIMARY_GROUP_COLS if c in df.columns]
    return [
        c for c in df.columns
        if c not in primary + metric_columns(df, "all")
        and c not in EXCLUDED_COLS
    ]


def primary_grouping_columns(df: pd.DataFrame) -> List[str]:
    return [c for c in PRIMARY_GROUP_COLS if c in df.columns]


def grouping_columns(df: pd.DataFrame) -> List[str]:
    return primary_grouping_columns(df) + hyperparam_columns(df)


def summarize_metrics(df: pd.DataFrame, metrics: List[str], group_cols: List[str]) -> pd.DataFrame:
    rows = []
    grouped = df.groupby(group_cols, dropna=False)
    for keys, part in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        if "bootstrap_iter" in part.columns:
            row["n_bootstrap"] = part["bootstrap_iter"].nunique(dropna=True)
        for metric in metrics:
            values = pd.to_numeric(part[metric], errors="coerce").dropna()
            if values.empty:
                row[f"{metric}_mean"] = pd.NA
                row[f"{metric}_p25"] = pd.NA
                row[f"{metric}_p75"] = pd.NA
            else:
                row[f"{metric}_mean"] = values.mean()
                row[f"{metric}_p25"] = values.quantile(0.25)
                row[f"{metric}_p75"] = values.quantile(0.75)
        rows.append(row)
    return pd.DataFrame(rows)


def format_summary(summary: pd.DataFrame, metrics: List[str]) -> pd.DataFrame:
    stat_suffixes = ("_mean", "_p25", "_p75")

    primary = [c for c in PRIMARY_GROUP_COLS if c in summary.columns]
    meta = [c for c in SUMMARY_META_COLS if c in summary.columns]
    hyperparams = sorted(
        c for c in summary.columns
        if c not in primary + meta + list(metrics)
        and not any(c.endswith(s) for s in stat_suffixes)
    )
    metric_cols = []
    for metric in metrics:
        for suffix in stat_suffixes:
            col = f"{metric}{suffix}"
            if col in summary.columns:
                metric_cols.append(col)

    display_cols = primary + meta + metric_cols + hyperparams
    out = summary[display_cols].copy()
    for col in metric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").round(4)
    return out


def _to_python_scalar(value):
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value


def build_results_dict(summary: pd.DataFrame, metrics: List[str]) -> dict:
    """Nested dict for copy-paste: {exp_name: {model_id: {...}}}."""
    stat_suffixes = ("_mean", "_p25", "_p75")
    primary = [
        c for c in PRIMARY_GROUP_COLS
        if c in summary.columns and c not in ("exp_name", "model_id")
    ]
    meta = [c for c in SUMMARY_META_COLS if c in summary.columns]
    hyperparams = sorted(
        c for c in summary.columns
        if c not in PRIMARY_GROUP_COLS and c not in meta + list(metrics)
        and not any(c.endswith(s) for s in stat_suffixes)
    )

    result = {}
    for _, row in summary.iterrows():
        exp_name = str(row["exp_name"]) if "exp_name" in summary.columns else "results"
        model_id = str(row["model_id"])

        entry = {"method": _to_python_scalar(row.get("method"))}
        for col in primary + meta:
            value = _to_python_scalar(row[col])
            if value is not None:
                entry[col] = value

        entry["metrics"] = {}
        for metric in metrics:
            entry["metrics"][metric] = {
                "mean": _to_python_scalar(row.get(f"{metric}_mean")),
                "p25": _to_python_scalar(row.get(f"{metric}_p25")),
                "p75": _to_python_scalar(row.get(f"{metric}_p75")),
            }

        hparams = {
            col: _to_python_scalar(row[col])
            for col in hyperparams
            if _to_python_scalar(row[col]) is not None
        }
        if hparams:
            entry["hparams"] = hparams

        result.setdefault(exp_name, {})[model_id] = entry

    return result

In [3]:
root = resolve_repo_path(EXPERIMENTS_PATH)
result_files = discover_result_files(root)
print(f"Найдено файлов результатов: {len(result_files)}")
for path in result_files:
    print(f"  - {path}")

raw = load_experiments(root)
if ONLY_SUCCESS and "error" in raw.columns:
    raw = raw[raw["error"].fillna(0).astype(int) == 0].copy()

metrics = metric_columns(raw, METRIC_SUFFIX)
group_cols = grouping_columns(raw)
primary_cols = primary_grouping_columns(raw)
hparam_cols = hyperparam_columns(raw)

print(f"\nСтрок после фильтрации: {len(raw)}")
print(f"Метрики: {metrics}")
print(f"Группировка: {primary_cols} + гиперпараметры {hparam_cols}")
raw.head()

Найдено файлов результатов: 1
  - /home/goverdovskiy/Backblaze_ML_Ops/Artifacts/Exp_166/grid_search.csv

Строк после фильтрации: 80
Метрики: ['ci_test', 'ci_train_max_size', 'ci_train_same_size', 'iauc_test', 'iauc_train_max_size', 'iauc_train_same_size', 'ibs_bal_test', 'ibs_bal_train_max_size', 'ibs_bal_train_same_size', 'ibs_test', 'ibs_train_max_size', 'ibs_train_same_size']
Группировка: ['exp_name', 'method', 'model_id', 'test_samples', 'score_samples', 'train_samples'] + гиперпараметры ['l1_ratio', 'learning_rate', 'max_depth', 'max_features', 'min_samples_split', 'n_estimators', 'penalizer', 'subsample']


,train_samples,method,model_id,train_time,test_time,error,error_text,test_samples,score_samples,bootstrap_iter,...,l1_ratio,learning_rate,max_depth,max_features,min_samples_split,n_estimators,penalizer,subsample,exp_name,result_file
0,1,CoxTV,0_CoxTV,223.886381,6.981018,0,NaN,25,10,0,...,0.01,NaN,NaN,NaN,NaN,NaN,0.01,NaN,Exp_166,grid_search.csv
1,1,CoxTV,0_CoxTV,223.886381,6.981018,0,NaN,25,10,1,...,0.01,NaN,NaN,NaN,NaN,NaN,0.01,NaN,Exp_166,grid_search.csv
2,1,CoxTV,0_CoxTV,223.886381,6.981018,0,NaN,25,10,2,...,0.01,NaN,NaN,NaN,NaN,NaN,0.01,NaN,Exp_166,grid_search.csv
3,1,CoxTV,0_CoxTV,223.886381,6.981018,0,NaN,25,10,3,...,0.01,NaN,NaN,NaN,NaN,NaN,0.01,NaN,Exp_166,grid_search.csv
4,1,CoxTV,0_CoxTV,223.886381,6.981018,0,NaN,25,10,4,...,0.01,NaN,NaN,NaN,NaN,NaN,0.01,NaN,Exp_166,grid_search.csv


In [4]:
summary = summarize_metrics(raw, metrics, group_cols)
summary_display = format_summary(summary, metrics)
summary_display

,exp_name,method,model_id,test_samples,score_samples,train_samples,n_bootstrap,ci_test_mean,ci_test_p25,ci_test_p75,...,ibs_train_same_size_p25,ibs_train_same_size_p75,l1_ratio,learning_rate,max_depth,max_features,min_samples_split,n_estimators,penalizer,subsample
0,Exp_166,CoxTILN,1_CoxTILN,25,10,1,10,0.8513,0.8458,0.8572,...,0.1036,0.1036,0.10,NaN,NaN,NaN,NaN,NaN,0.10,NaN
1,Exp_166,CoxTILN,5_CoxTILN,25,10,20,10,0.8418,0.8365,0.8450,...,0.1536,0.1536,0.10,NaN,NaN,NaN,NaN,NaN,0.10,NaN
2,Exp_166,CoxTV,0_CoxTV,25,10,1,10,0.8192,0.8149,0.8260,...,0.1021,0.1021,0.01,NaN,NaN,NaN,NaN,NaN,0.01,NaN
3,Exp_166,CoxTV,4_CoxTV,25,10,20,10,0.8283,0.8227,0.8314,...,0.2103,0.2103,0.01,NaN,NaN,NaN,NaN,NaN,0.01,NaN
4,Exp_166,GBSA,3_GBSA,25,10,1,10,0.9005,0.8977,0.9055,...,0.0582,0.0582,NaN,0.1,4.0,NaN,NaN,200.0,NaN,0.8
5,Exp_166,GBSA,7_GBSA,25,10,20,10,0.8937,0.8907,0.8983,...,0.1134,0.1134,NaN,0.1,4.0,NaN,NaN,50.0,NaN,0.8
6,Exp_166,RSF,2_RSF,25,10,1,10,0.9035,0.9021,0.9077,...,0.0615,0.0615,NaN,NaN,8.0,0.7,10.0,50.0,NaN,NaN
7,Exp_166,RSF,6_RSF,25,10,20,10,0.9036,0.9016,0.9076,...,0.0954,0.0954,NaN,NaN,8.0,0.7,10.0,50.0,NaN,NaN


In [5]:
# Компактный вид: одна строка на модель, метрики в формате mean [p25, p75]
hparam_cols_display = sorted(
    c for c in summary_display.columns
    if c not in PRIMARY_GROUP_COLS and c not in SUMMARY_META_COLS
    and c not in metrics
    and not any(c.endswith(s) for s in ("_mean", "_p25", "_p75"))
)

compact_rows = []
for _, row in summary_display.iterrows():
    compact = {}
    for c in [x for x in PRIMARY_GROUP_COLS if x in summary_display.columns]:
        compact[c] = row[c]
    for c in [x for x in SUMMARY_META_COLS if x in summary_display.columns]:
        compact[c] = row[c]
    for metric in metrics:
        mean = row.get(f"{metric}_mean")
        p25 = row.get(f"{metric}_p25")
        p75 = row.get(f"{metric}_p75")
        if pd.isna(mean):
            compact[metric] = None
        else:
            compact[metric] = f"{mean:.4f} [{p25:.4f}, {p75:.4f}]"
    for c in hparam_cols_display:
        compact[c] = row[c]
    compact_rows.append(compact)

pd.DataFrame(compact_rows)

,exp_name,method,model_id,test_samples,score_samples,train_samples,n_bootstrap,ci_test,ci_train_max_size,ci_train_same_size,...,ibs_train_max_size,ibs_train_same_size,l1_ratio,learning_rate,max_depth,max_features,min_samples_split,n_estimators,penalizer,subsample
0,Exp_166,CoxTILN,1_CoxTILN,25,10,1,10,"0.8513 [0.8458, 0.8572]","0.8420 [0.8420, 0.8420]","0.8276 [0.8276, 0.8276]",...,"0.2117 [0.2117, 0.2117]","0.1036 [0.1036, 0.1036]",0.10,NaN,NaN,NaN,NaN,NaN,0.10,NaN
1,Exp_166,CoxTILN,5_CoxTILN,25,10,20,10,"0.8418 [0.8365, 0.8450]","0.8416 [0.8416, 0.8416]","0.8416 [0.8416, 0.8416]",...,"0.1536 [0.1536, 0.1536]","0.1536 [0.1536, 0.1536]",0.10,NaN,NaN,NaN,NaN,NaN,0.10,NaN
2,Exp_166,CoxTV,0_CoxTV,25,10,1,10,"0.8192 [0.8149, 0.8260]","0.7623 [0.7623, 0.7623]","0.8043 [0.8043, 0.8043]",...,"0.3902 [0.3902, 0.3902]","0.1021 [0.1021, 0.1021]",0.01,NaN,NaN,NaN,NaN,NaN,0.01,NaN
3,Exp_166,CoxTV,4_CoxTV,25,10,20,10,"0.8283 [0.8227, 0.8314]","0.8389 [0.8389, 0.8389]","0.8389 [0.8389, 0.8389]",...,"0.2103 [0.2103, 0.2103]","0.2103 [0.2103, 0.2103]",0.01,NaN,NaN,NaN,NaN,NaN,0.01,NaN
4,Exp_166,GBSA,3_GBSA,25,10,1,10,"0.9005 [0.8977, 0.9055]","0.6991 [0.6991, 0.6991]","0.9178 [0.9178, 0.9178]",...,"0.3011 [0.3011, 0.3011]","0.0582 [0.0582, 0.0582]",NaN,0.1,4.0,NaN,NaN,200.0,NaN,0.8
5,Exp_166,GBSA,7_GBSA,25,10,20,10,"0.8937 [0.8907, 0.8983]","0.8945 [0.8945, 0.8945]","0.8945 [0.8945, 0.8945]",...,"0.1134 [0.1134, 0.1134]","0.1134 [0.1134, 0.1134]",NaN,0.1,4.0,NaN,NaN,50.0,NaN,0.8
6,Exp_166,RSF,2_RSF,25,10,1,10,"0.9035 [0.9021, 0.9077]","0.7629 [0.7629, 0.7629]","0.9169 [0.9169, 0.9169]",...,"0.1552 [0.1552, 0.1552]","0.0615 [0.0615, 0.0615]",NaN,NaN,8.0,0.7,10.0,50.0,NaN,NaN
7,Exp_166,RSF,6_RSF,25,10,20,10,"0.9036 [0.9016, 0.9076]","0.9092 [0.9092, 0.9092]","0.9092 [0.9092, 0.9092]",...,"0.0954 [0.0954, 0.0954]","0.0954 [0.0954, 0.0954]",NaN,NaN,8.0,0.7,10.0,50.0,NaN,NaN


## Словарь для копирования

Структура: `{exp_name: {model_id: {method, ..., metrics: {ci_test: {mean, p25, p75}, ...}}}}`

Скопируй строку из output или переменную `results_dict`.

In [6]:
import json

results_dict = build_results_dict(summary_display, metrics)
print(json.dumps(results_dict, indent=2, ensure_ascii=False))
results_dict

{
  "Exp_166": {
    "1_CoxTILN": {
      "method": "CoxTILN",
      "test_samples": 25,
      "score_samples": 10,
      "train_samples": 1,
      "n_bootstrap": 10,
      "metrics": {
        "ci_test": {
          "mean": 0.8513,
          "p25": 0.8458,
          "p75": 0.8572
        },
        "ci_train_max_size": {
          "mean": 0.842,
          "p25": 0.842,
          "p75": 0.842
        },
        "ci_train_same_size": {
          "mean": 0.8276,
          "p25": 0.8276,
          "p75": 0.8276
        },
        "iauc_test": {
          "mean": 0.6784,
          "p25": 0.6725,
          "p75": 0.6845
        },
        "iauc_train_max_size": {
          "mean": 0.7848,
          "p25": 0.7848,
          "p75": 0.7848
        },
        "iauc_train_same_size": {
          "mean": 0.7917,
          "p25": 0.7917,
          "p75": 0.7917
        },
        "ibs_bal_test": {
          "mean": 0.2103,
          "p25": 0.2063,
          "p75": 0.2125
        },
        "ibs_ba

{'Exp_166': {'1_CoxTILN': {'method': 'CoxTILN',
   'test_samples': 25,
   'score_samples': 10,
   'train_samples': 1,
   'n_bootstrap': 10,
   'metrics': {'ci_test': {'mean': 0.8513, 'p25': 0.8458, 'p75': 0.8572},
    'ci_train_max_size': {'mean': 0.842, 'p25': 0.842, 'p75': 0.842},
    'ci_train_same_size': {'mean': 0.8276, 'p25': 0.8276, 'p75': 0.8276},
    'iauc_test': {'mean': 0.6784, 'p25': 0.6725, 'p75': 0.6845},
    'iauc_train_max_size': {'mean': 0.7848, 'p25': 0.7848, 'p75': 0.7848},
    'iauc_train_same_size': {'mean': 0.7917, 'p25': 0.7917, 'p75': 0.7917},
    'ibs_bal_test': {'mean': 0.2103, 'p25': 0.2063, 'p75': 0.2125},
    'ibs_bal_train_max_size': {'mean': 0.2404, 'p25': 0.2404, 'p75': 0.2404},
    'ibs_bal_train_same_size': {'mean': 0.1769, 'p25': 0.1769, 'p75': 0.1769},
    'ibs_test': {'mean': 0.2162, 'p25': 0.2148, 'p75': 0.2178},
    'ibs_train_max_size': {'mean': 0.2117, 'p25': 0.2117, 'p75': 0.2117},
    'ibs_train_same_size': {'mean': 0.1036, 'p25': 0.1036, 'p75

In [7]:
# Сводка по методам внутри каждого эксперимента (усреднение по model_id)
method_group = [c for c in ("exp_name", "method", "test_samples", "score_samples") if c in raw.columns]
method_summary = summarize_metrics(raw, metrics, method_group)
format_summary(method_summary, metrics)

,exp_name,method,test_samples,score_samples,n_bootstrap,ci_test_mean,ci_test_p25,ci_test_p75,ci_train_max_size_mean,ci_train_max_size_p25,...,ibs_bal_train_same_size_p75,ibs_test_mean,ibs_test_p25,ibs_test_p75,ibs_train_max_size_mean,ibs_train_max_size_p25,ibs_train_max_size_p75,ibs_train_same_size_mean,ibs_train_same_size_p25,ibs_train_same_size_p75
0,Exp_166,CoxTILN,25,10,10,0.8466,0.8435,0.8518,0.8418,0.8416,...,0.1945,0.1815,0.1474,0.2158,0.1827,0.1536,0.2117,0.1286,0.1036,0.1536
1,Exp_166,CoxTV,25,10,10,0.8238,0.8196,0.8300,0.8006,0.7623,...,0.2392,0.2181,0.2163,0.2208,0.3003,0.2103,0.3902,0.1562,0.1021,0.2103
2,Exp_166,GBSA,25,10,10,0.8971,0.8912,0.9040,0.7968,0.6991,...,0.1474,0.1294,0.1109,0.1455,0.2073,0.1134,0.3011,0.0858,0.0582,0.1134
3,Exp_166,RSF,25,10,10,0.9036,0.9010,0.9077,0.8361,0.7629,...,0.1213,0.1224,0.0973,0.1448,0.1253,0.0954,0.1552,0.0785,0.0615,0.0954
